# RPO

- Arbitrary RAAN, inclination, and phasing
- Angles, then 3 DOF, then 6 DOF

In [334]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [362]:
import numpy as np
from cesium.constants import DEG_TO_RAD, KM_TO_M, M_TO_KM, RAD_TO_DEG, SEC_TO_MIN, SEC_TO_HOUR, SEC_TO_DAY, DAY_TO_SEC
from cesium.guidance.coplanar import guid_coaxial_hohmann, guid_phasing_same_orbit
from cesium.guidance.noncoplanar import guid_combined_i_raan_change
from cesium.math.base import angle_between, unit
from cesium.math.rotations import wrap_to_2pi, wrap_to_pi
from cesium.orbital.coes import (
    COES,
    E_from_ta,
    kepler_eq,
    ta_from_E,
)
from cesium.orbital.transformations import calc_dcm_rsw_wrt_pci
from cesium.physics.dynamics import propagate_two_body
from cesium.plotting import plot_3d
from cesium.world.constants import J2_MARS, MU_MARS, R_MARS
from numpy.linalg import norm
import plotly.graph_objects as go
from tqdm import tqdm

## Setup
Assume within 5 degrees inclination, raan, 0.05 e, 100km a

In [336]:
# Target orbit (same as arrival.ipynb), MCI, circular
PARKING_ORBIT_ALT_M = 300e3  # m
PARKING_ORBIT_RAAN_DEG = 15
PARKING_ORBIT_I_DEG = 45
PARKING_ORBIT_TA_DEG = 13

TARGET_0 = COES(
    a=R_MARS + PARKING_ORBIT_ALT_M,
    e=0.0,
    i=PARKING_ORBIT_I_DEG * DEG_TO_RAD,
    raan=PARKING_ORBIT_RAAN_DEG * DEG_TO_RAD,
    aop=0.0,
    ta=PARKING_ORBIT_TA_DEG * DEG_TO_RAD,
)
a_t, e_t, i_t, raan_t, aop_t, ta_t = TARGET_0

dt = 1 # sec

In [337]:
# Chaser = target + seeded uniform dispersion
SEED = 525600
# SEED = 13
ANGLE_DISP_DEG = 3.0  # i, raan, aop
A_DISP_M = 100 * KM_TO_M
E_MAX = 0.01  # e >= 0, so draw from [0, E_MAX]
J2 = 0
INITIAL_TA_OFFSET_DEG = 57
J2 = J2_MARS

# CRAZY
# ANGLE_DISP_DEG = 50.0  # i, raan, aop
# A_DISP_M = 500 * KM_TO_M

rng = np.random.default_rng(SEED)
rel_coes = np.array(
    [
        rng.uniform(-A_DISP_M, A_DISP_M),
        rng.uniform(0.0, E_MAX),
        *rng.uniform(-ANGLE_DISP_DEG, ANGLE_DISP_DEG, 3) * DEG_TO_RAD,
        INITIAL_TA_OFFSET_DEG * DEG_TO_RAD,  # ta: phasing left to the RPO problem
    ]
)
CHASER_0 = COES.from_array(TARGET_0.elements() + rel_coes)
a_c, e_c, i_c, raan_c, aop_c, ta_c = CHASER_0

TARGET_0.print("initial target coes")
CHASER_0.print("initial chaser coes")

# mean motion
n_c0 = np.sqrt(MU_MARS / a_c**3)
n_t0 = np.sqrt(MU_MARS / a_t**3)

initial target coes: a = 3696.000 km, e = 0.00000, i = 45.000 deg, raan = 15.000 deg, aop = 0.000 deg, ta = 13.000 deg
initial chaser coes: a = 3754.914 km, e = 0.00763, i = 46.710 deg, raan = 15.208 deg, aop = 2.930 deg, ta = 70.000 deg


In [338]:
# Propagate both for a few target periods
N_ORBITS = 3
period_s = 2 * np.pi * np.sqrt(a_t**3 / MU_MARS)
t_s = np.linspace(0, N_ORBITS * period_s, 3000)

t0 = 0
rv_target_mci = propagate_two_body(TARGET_0.to_rv(MU_MARS), t_s, mu=MU_MARS, j2=J2, r_cb=R_MARS)
rv_chaser_mci = propagate_two_body(CHASER_0.to_rv(MU_MARS), t_s, mu=MU_MARS, j2=J2, r_cb=R_MARS)

## Planning
- long way
  - raan and i
  - circularize
  - phasing
- short way
  - minimum combined plane change (but figure that out)

**Timings**
- *Time 0:* Start of trajectory
- *Time 1:* (Burn 1) RAAN/i burn at line of nodes
- *Time 2:* (Burn 2) Periapse
- *Time 3:* (Burn 3) Circularization
- *Time 4:* (Burn 4) Phasing 1
- *Time 5:* (Burn 5) Phasing 2

**Larger plan**
1. Line of nodes
2. Find time until that node
3. Burn at `t1` at that node in specified direction for `i` and `raan` change
4. Recalculate new coes

In [339]:
def node_line_finder(rv_chaser: np.ndarray, rv_target: np.ndarray):
    r_c,v_c = rv_chaser[0, 0:3], rv_chaser[0, 3:6]
    h_chaser = np.cross(r_c, v_c)
    h_target = np.cross(rv_target[0, 0:3], rv_target[0, 3:6])
    n_hat = unit(np.cross(h_chaser, h_target))

    d_ta = angle_between(r_c, n_hat) * np.sign(
        np.dot(v_c, n_hat)
    )  # change sign if further in orbit to make it signed
    d_ta = wrap_to_2pi(d_ta)

    print(f"Angle until node line: {d_ta * RAD_TO_DEG}º")
    
    return n_hat, d_ta, h_chaser, h_target

# Target position at the burn
def M_from_ta(ta,e):
    E = E_from_ta(ta, e)
    M = E - e*np.sin(E)
    return M

def ta_from_M(M,e):
    E = kepler_eq(M,e)
    ta = ta_from_E(E,e)
    return ta

def orbit_sweep(coes: COES, d_ta_rad: float, mu: float):
    M = M_from_ta(coes.ta, coes.e)
    ta_new = coes.ta + d_ta_rad
    M_new = M_from_ta(ta_new, coes.e)

    dM = wrap_to_2pi(M_new - M)

    t_coast = dM / coes.n(mu)

    return coes.with_ta(ta_new), t_coast

def pkepler_coes(coes: COES, dt: float, mu: float):
    M = M_from_ta(coes.ta, coes.e)
    M_new = M + dt*coes.n(mu)
    ta_new = ta_from_M(M_new, coes.e)
    return coes.with_ta(ta_new)

def apply_dv(coes: COES, dv: np.ndarray, mu: float):
    rv = coes.to_rv(mu)
    return COES.from_rv(rv[0:3], rv[3:6] + dv, mu)

### From 0 -> 1

In [340]:
# Burn length
n_hat, d_ta, h_chaser, h_target = node_line_finder(rv_chaser_mci, rv_target_mci)

CHASER_1, dt_0_to_1 = orbit_sweep(CHASER_0, d_ta, MU_MARS)
TARGET_1 = pkepler_coes(TARGET_0, dt_0_to_1, MU_MARS)

print(f"Coast time: {dt_0_to_1 * SEC_TO_MIN:.5f} min")
t1 = t0 + dt_0_to_1

ts_0_to_1 = np.linspace(t0, t1, 3000)
rv_c_0_to_1 = propagate_two_body(CHASER_0.to_rv(MU_MARS), ts_0_to_1, mu=MU_MARS, j2=J2, r_cb=R_MARS)
rv_t_0_to_1 = propagate_two_body(TARGET_0.to_rv(MU_MARS), ts_0_to_1, mu=MU_MARS, j2=J2, r_cb=R_MARS)

Angle until node line: 111.9924342775552º
Coast time: 36.49519 min


### Time 1:  Burn 1

In [341]:
# Burn
dv_i_raan = guid_combined_i_raan_change(
    CHASER_0.i,
    TARGET_0.i,
    TARGET_0.raan - CHASER_0.raan,
    norm(CHASER_0.to_rv(MU_MARS)[3:6]),
)
print(f"dv1 = {dv_i_raan:.3f} m/s")

dv_burn1 = dv_i_raan * unit(h_chaser)

CHASER_1A = apply_dv(CHASER_1, dv_burn1, MU_MARS)

dv1 = 101.461 m/s


In [342]:

# dv_burn1_rsw = calc_dcm_rsw_wrt_pci(CHASER_1.to_rv(MU_MARS)) @ dv_burn1
# print(
#     f"Burn 1 at t = {t1 * SEC_TO_MIN:.2f} min (node, {d_ta * RAD_TO_DEG:.2f} deg down orbit from t0)"
# )
# print(
#     f"  |dv| = {norm(dv_burn1):.3f} m/s; RSW = [{dv_burn1_rsw[0]:.3f}, {dv_burn1_rsw[1]:.3f}, {dv_burn1_rsw[2]:.3f}] m/s"
# )
# print()

def print_diffs(coes_t: COES, coes_c1: COES, coes_c2: COES, mu: float = MU_MARS):
    coes_t.print("target            ", mu)
    coes_c1.print("chaser before     ", mu)
    coes_c2.print("chaser after      ", mu)
    coes_c2.delta(coes_c1).print("change            ", mu)
    print()
    coes_c1.delta(coes_t).print("before - target   ", mu)
    coes_c2.delta(coes_t).print("after - target    ", mu)
    print()

print_diffs(TARGET_1, CHASER_1, CHASER_1A, MU_MARS)

target            : a = 3696.000 km, e = 0.00000, i = 45.000 deg, raan = 15.000 deg, aop = 0.000 deg, ta = 128.551 deg (E_sp = -5.794 km2/s2)
chaser before     : a = 3754.914 km, e = 0.00763, i = 46.710 deg, raan = 15.208 deg, aop = 2.930 deg, ta = 181.992 deg (E_sp = -5.703 km2/s2)
chaser after      : a = 3758.306 km, e = 0.00672, i = 44.983 deg, raan = 14.998 deg, aop = 2.806 deg, ta = 182.263 deg (E_sp = -5.698 km2/s2)
change            : a = 3.392 km, e = -0.00091, i = -1.727 deg, raan = -0.210 deg, aop = -0.124 deg, ta = 0.271 deg (E_sp = -6312.985 km2/s2)

before - target   : a = 58.914 km, e = 0.00763, i = 1.710 deg, raan = 0.208 deg, aop = 2.930 deg, ta = 53.441 deg (E_sp = -363.480 km2/s2)
after - target    : a = 62.306 km, e = 0.00672, i = -0.017 deg, raan = -0.002 deg, aop = 2.806 deg, ta = 53.712 deg (E_sp = -343.691 km2/s2)



### From 1 -> 2

In [343]:
# Very circular, so we should just depart at whichever is sooner
periapsis_is_closer = CHASER_1A.ta > np.pi
print(f"{periapsis_is_closer = }")

d_ta_chaser_1_to_2 = (2*np.pi - CHASER_1A.ta) if periapsis_is_closer else wrap_to_2pi(np.pi - CHASER_1A.ta)
CHASER_2, dt_1_to_2 = orbit_sweep(CHASER_1A, d_ta_chaser_1_to_2, MU_MARS)
TARGET_2 = pkepler_coes(TARGET_1, dt_1_to_2, MU_MARS)

# Propagate 1 to 2
t2 = t1 + dt_1_to_2
ts_1_to_2 = np.linspace(t1, t2, 3000)
rv_c_1_to_2 = propagate_two_body(CHASER_1A.to_rv(MU_MARS), ts_1_to_2, mu=MU_MARS, j2=J2, r_cb=R_MARS)
rv_t_1_to_2 = propagate_two_body(TARGET_1.to_rv(MU_MARS), ts_1_to_2, mu=MU_MARS, j2=J2, r_cb=R_MARS)

ts_0_to_2 = np.concatenate((ts_0_to_1[:-1], ts_1_to_2))
rv_c_0_to_2 = np.concatenate((rv_c_0_to_1[:-1], rv_c_1_to_2))
rv_t_0_to_2 = np.concatenate((rv_t_0_to_1[:-1], rv_t_1_to_2))

periapsis_is_closer = True


### Time 2-3: Hohmann burns

In [344]:
a_trans, dt_2_to_3, dv_a, dv_b = guid_coaxial_hohmann(
    CHASER_2.r_p, CHASER_2.r_a, TARGET_2.r_p, TARGET_2.r_a, depart_at_periapsis=periapsis_is_closer, mu=MU_MARS
)

print(f"sma of transfer orbit: {a_trans} m")
print(f"Transfer orbit (half) period {dt_2_to_3 * SEC_TO_MIN} min")
print(f"{dv_a = :.3f} m/s (negative means retrograde)")
print(f"{dv_b = :.3f} m/s (negative means retrograde)")

sma of transfer orbit: 3714527.9760046853 m
Transfer orbit (half) period 57.27830737415616 min
dv_a = -19.817 m/s (negative means retrograde)
dv_b = -8.479 m/s (negative means retrograde)


In [345]:
# Burning chaser
dv_burn2 = dv_a * unit(CHASER_2.to_rv(MU_MARS)[3:6])
CHASER_2A = apply_dv(CHASER_2, dv_burn2, MU_MARS)

print_diffs(TARGET_2, CHASER_2, CHASER_2A, MU_MARS)

# States at 3
CHASER_3 = pkepler_coes(CHASER_2A, dt_2_to_3, MU_MARS) # SHOULD BE THE SAME AS ORBIT SWEEPING
TARGET_3 = pkepler_coes(TARGET_2, dt_2_to_3, MU_MARS)

# Burning chaser
dv_burn3 = dv_b * unit(CHASER_3.to_rv(MU_MARS)[3:6])
CHASER_3A = apply_dv(CHASER_3, dv_burn3, MU_MARS)

print_diffs(TARGET_3, CHASER_3, CHASER_3A, MU_MARS)


target            : a = 3696.000 km, e = 0.00000, i = 45.000 deg, raan = 15.000 deg, aop = 0.000 deg, ta = 310.770 deg (E_sp = -5.794 km2/s2)
chaser before     : a = 3758.306 km, e = 0.00672, i = 44.983 deg, raan = 14.998 deg, aop = 2.806 deg, ta = 360.000 deg (E_sp = -5.698 km2/s2)
chaser after      : a = 3714.528 km, e = 0.00499, i = 44.983 deg, raan = 14.998 deg, aop = 182.806 deg, ta = 180.000 deg (E_sp = -5.765 km2/s2)
change            : a = -43.778 km, e = -0.00173, i = 0.000 deg, raan = 0.000 deg, aop = -180.000 deg, ta = 180.000 deg (E_sp = 489.151 km2/s2)

before - target   : a = 62.306 km, e = 0.00672, i = -0.017 deg, raan = -0.002 deg, aop = 2.806 deg, ta = 49.230 deg (E_sp = -343.691 km2/s2)
after - target    : a = 18.528 km, e = 0.00499, i = -0.017 deg, raan = -0.002 deg, aop = -177.194 deg, ta = -130.770 deg (E_sp = -1155.766 km2/s2)

target            : a = 3696.000 km, e = 0.00000, i = 45.000 deg, raan = 15.000 deg, aop = 0.000 deg, ta = 132.125 deg (E_sp = -5.794 km2/

### From 2 -> 3

In [346]:
# Propagate 2 to 3
t3 = t2 + dt_2_to_3
ts_2_to_3 = np.linspace(t2, t3, 3000)
rv_c_2_to_3 = propagate_two_body(CHASER_2A.to_rv(MU_MARS), ts_2_to_3, MU_MARS)
rv_t_2_to_3 = propagate_two_body(TARGET_2.to_rv(MU_MARS), ts_2_to_3, MU_MARS)

ts_0_to_3 = np.concatenate((ts_0_to_2[:-1], ts_2_to_3))
rv_c_0_to_3 = np.concatenate((rv_c_0_to_2[:-1], rv_c_2_to_3))
rv_t_0_to_3 = np.concatenate((rv_t_0_to_2[:-1], rv_t_2_to_3))

### From 3 -> 4

In [347]:
# Nothing
dt_3_to_4 = 0
t4 = t3 + dt_3_to_4

CHASER_4 = CHASER_3A.copy()
TARGET_4 = TARGET_3.copy()

ts_0_to_4 = ts_0_to_3.copy()
rv_c_0_to_4 = rv_c_0_to_3.copy()
rv_t_0_to_4 = rv_t_0_to_3.copy()


### Time 4-5: Phasing entry

In [348]:
# I'm kinda impatient
k_target = 20
k_chaser = 20


d_ta = wrap_to_pi(CHASER_4.aol - TARGET_4.aol)
dv_4_5_tot, dt_4_to_5, a_phase = guid_phasing_same_orbit(TARGET_4.a, d_ta, MU_MARS, k_target, k_chaser, deg=False, r_cb=R_MARS)

print(f"Orbital separation: {d_ta*RAD_TO_DEG:.5f}º")
print(f"sma of transfer orbit: {a_phase*M_TO_KM} m (compared to target {TARGET_4.a*M_TO_KM:.2f} km and {CHASER_4.a*M_TO_KM:.2f} km)")
print(f"Transfer duration {dt_4_to_5 * SEC_TO_MIN} min ({dt_4_to_5 * SEC_TO_HOUR} hr, {dt_4_to_5 * SEC_TO_DAY} day))")
print(f"{dv_4_5_tot = :.3f} m/s (negative means retrograde)")

Orbital separation: 50.68045º
sma of transfer orbit: 3713.3236907979617 m (compared to target 3696.00 km and 3696.00 km)
Transfer duration 2290.0181750441216 min (38.166969584068696 hr, 1.5902903993361956 day))
dv_4_5_tot = 15.862 m/s (negative means retrograde)


In [349]:
# Burning chaser
dv_4_5 = dv_4_5_tot / 2 * unit(CHASER_4.to_rv(MU_MARS)[3:6]) 
CHASER_4A = apply_dv(CHASER_4, dv_4_5, MU_MARS)

print_diffs(TARGET_4, CHASER_4, CHASER_4A, MU_MARS)

CHASER_5 = pkepler_coes(CHASER_4A, dt_4_to_5, MU_MARS) # SHOULD BE THE SAME AS ORBIT SWEEPING
TARGET_5 = pkepler_coes(TARGET_4, dt_4_to_5, MU_MARS)

# Burning chaser
CHASER_5A = apply_dv(CHASER_5, -dv_4_5, MU_MARS) # Since burning back now

print_diffs(TARGET_5, CHASER_5, CHASER_5A, MU_MARS)

target            : a = 3696.000 km, e = 0.00000, i = 45.000 deg, raan = 15.000 deg, aop = 0.000 deg, ta = 132.125 deg (E_sp = -5.794 km2/s2)
chaser before     : a = 3696.000 km, e = 0.00000, i = 44.983 deg, raan = 14.998 deg, aop = 0.000 deg, ta = 182.806 deg (E_sp = -5.794 km2/s2)
chaser after      : a = 3713.324 km, e = 0.00467, i = 44.983 deg, raan = 14.998 deg, aop = 182.806 deg, ta = 360.000 deg (E_sp = -5.767 km2/s2)
change            : a = 17.324 km, e = 0.00467, i = 0.000 deg, raan = 0.000 deg, aop = -177.194 deg, ta = 177.194 deg (E_sp = -1236.111 km2/s2)

before - target   : a = -0.000 km, e = 0.00000, i = -0.017 deg, raan = -0.002 deg, aop = 0.000 deg, ta = 50.680 deg
after - target    : a = 17.324 km, e = 0.00467, i = -0.017 deg, raan = -0.002 deg, aop = -177.194 deg, ta = -132.125 deg (E_sp = -1236.111 km2/s2)

target            : a = 3696.000 km, e = 0.00000, i = 45.000 deg, raan = 15.000 deg, aop = 0.000 deg, ta = 182.806 deg (E_sp = -5.794 km2/s2)
chaser before     : a

### From 4 -> 5

In [350]:
# Propagate 2 to 3
t5 = t4 + dt_4_to_5
ts_4_to_5 = np.linspace(t4, t5, 3000)
rv_c_4_to_5 = propagate_two_body(CHASER_4A.to_rv(MU_MARS), ts_4_to_5, MU_MARS)
rv_t_4_to_5 = propagate_two_body(TARGET_4.to_rv(MU_MARS), ts_4_to_5, MU_MARS)

ts_0_to_5 = np.concatenate((ts_0_to_4[:-1], ts_4_to_5))
rv_c_0_to_5 = np.concatenate((rv_c_0_to_4[:-1], rv_c_4_to_5))
rv_t_0_to_5 = np.concatenate((rv_t_0_to_4[:-1], rv_t_4_to_5))

### Onward

In [ ]:
ts_5_to_6 = 500*DAY_TO_SEC
t6 = t5 + ts_5_to_6
# ts_5_to_6 = np.linspace(t5, t6, 3000)
ts_5_to_6 = np.linspace(t5, t6, 300_000) # will take like a minute to run (GOOD FOR VERIFYING THAT IT DOESN'T DRIFT)
rv_c_5_to_6 = propagate_two_body(CHASER_5A.to_rv(MU_MARS), ts_5_to_6, MU_MARS)
rv_t_5_to_6 = propagate_two_body(TARGET_5.to_rv(MU_MARS), ts_5_to_6, MU_MARS)

ts_0_to_6 = np.concatenate((ts_0_to_5[:-1], ts_5_to_6))
rv_c_0_to_6 = np.concatenate((rv_c_0_to_5[:-1], rv_c_5_to_6))
rv_t_0_to_6 = np.concatenate((rv_t_0_to_5[:-1], rv_t_5_to_6))

In [359]:
def plot_separation(ts, r_c, r_t, burns):

    # Chaser-to-target separation over the whole plan
    separation_km = norm(r_c[:, 0:3] - r_t[:, 0:3], axis=1) * M_TO_KM
    t_min = ts * SEC_TO_MIN

    fig_sep = go.Figure(
        go.Scatter(
            x=t_min,
            y=separation_km,
            mode="lines",
            line={"color": "#2a78d6", "width": 2},
            hovertemplate="t = %{x:.1f} min<br>separation = %{y:.1f} km<extra></extra>",
        )
    )
    for t_burn, name in burns:
        fig_sep.add_vline(
            x=t_burn * SEC_TO_MIN,
            line={"color": "#898781", "width": 1, "dash": "dot"},
            annotation_text=name,
            annotation_position="top",
            annotation_font={"color": "#52514e", "size": 11},
        )
    fig_sep.update_layout(
        title="Chaser-to-target separation",
        xaxis_title="Time [min]",
        yaxis_title="Separation [km]",
        template="plotly_white",
        showlegend=False,
        height=420,
        margin={"t": 70},
    )
    fig_sep.update_yaxes(rangemode="tozero")
    fig_sep.show()

burns = [
    (t1, "Burn 1 (plane)"),
    (t2, "Burn 2 (Hohmann)"),
    (t3, "Burn 3 (circularize)"),
    (t4, "Burn 4 (Phase)"),
    (t5, "Burn 5 (circularize)"),
]

plot_separation(ts_0_to_6, rv_c_0_to_6, rv_t_0_to_6, burns)

### RSW

In [364]:
rv_c_5_to_6_rsw = np.zeros_like(rv_c_5_to_6)

dcms_rsw_wrt_pci = np.zeros((rv_c_5_to_6.shape[0], 3, 3))


for i in tqdm(range(rv_c_5_to_6.shape[0])):
    dcms_rsw_wrt_pci[i] = calc_dcm_rsw_wrt_pci(rv_t_5_to_6[i])
    rv_c_5_to_6_rsw[i] = dcms_rsw_wrt_pci[i] @ rv_c_5_to_6[i]

  0%|          | 0/300000 [00:00<?, ?it/s]


ValueError: matmul: Input operand 1 has a mismatch in its core dimension 0, with gufunc signature (n?,k),(k,m?)->(n?,m?) (size 6 is different from 3)

## Visualization

In [360]:
vectors = [
    {
        "direction": n_hat,
        "length": 20000,
        "name": "Node line",
        "color": "goldenrod",
        "label": True,
    },
]

rs = [
    # rv_t_0_to_1[:, 0:3] * M_TO_KM,
    # rv_c_0_to_1[:, 0:3] * M_TO_KM,
    # rv_t_0_to_3[:, 0:3] * M_TO_KM,
    # rv_c_0_to_3[:, 0:3] * M_TO_KM,
    # rv_t_0_to_3[:, 0:3] * M_TO_KM,
    # rv_c_0_to_3[:, 0:3] * M_TO_KM,
    # rv_c_0_to_2[:, 0:3] * M_TO_KM,
    rv_t_0_to_6[:, 0:3] * M_TO_KM,
    rv_c_0_to_6[:, 0:3] * M_TO_KM,
]
ts = [
    # ts_0_to_1,
    # ts_0_to_1,
    # ts_0_to_3,
    # ts_0_to_3,
    ts_0_to_6,
    ts_0_to_6,
]
labels=[
    "Target",
    "Chaser"
]
trace_colors=[
    "black",
    "royalblue"
]
points = [
    {
        "name": "r_t1",
        "location": rv_t_0_to_5[-1,0:3] * M_TO_KM,
        "color": "blue",
    },
    {
        "name": "r_c1",
        "location": rv_c_0_to_5[-1,0:3] * M_TO_KM,
        "color": "yellow",
    },
]

fig = plot_3d(
    rs, ts, labels, trace_colors=trace_colors,
    bodies={"name": "Mars", "radius": R_MARS * M_TO_KM, "color": "orangered"},
    title=f"Target and chaser, {N_ORBITS} orbits (MCI)",
    vectors=vectors,
    points=points,
    units="km",
)

In [ ]:
vectors = [
    {
        "direction": n_hat,
        "length": 20000,
        "name": "Node line",
        "color": "goldenrod",
        "label": True,
    },
]

points = [
    # {
    #     "name": "r_t1",
    #     "location": rv_t_0_to_1[-1,0:3] * M_TO_KM,
    #     "color": "blue",
    # },
    # {
    #     "name": "r_c1",
    #     "location": rv_c_0_to_1[-1,0:3] * M_TO_KM,
    #     "color": "yellow",
    # },
]

fig = plot_3d(
    [rv_t_0_to_1[:,0:3] * M_TO_KM, rv_c_0_to_1[:,0:3] * M_TO_KM],
    [ts_0_to_1, ts_0_to_1],
    labels=["Target", "Chaser"],
    trace_colors=["purple", "green"],
    bodies={"name": "Mars", "radius": R_MARS * M_TO_KM, "color": "orangered"},
    title=f"Target and chaser, {N_ORBITS} orbits (MCI)",
    vectors=vectors,
    points=points,
    units="km",
)